## 02 Prediction

Train simple baselines for 30-second mid-price direction prediction using chronological splits.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.metrics import evaluate_classifier, evaluate_predictions, rule_based_obi_predict
from src.models import (
    get_feature_target_data,
    time_series_train_val_test_split,
    train_logistic_regression,
    train_xgboost_classifier,
)

In [2]:
data_path = PROJECT_ROOT / "data" / "processed" / "obi_dataset.parquet"
df = pd.read_parquet(data_path).sort_values("timestamp").reset_index(drop=True)
df[["timestamp", "mid_price", "obi_5", "future_return_30s", "label"]].head()

,timestamp,mid_price,obi_5,future_return_30s,label
0,2021-04-07 11:32:42.122161+00:00,56035.995,0.175507,-0.00301,Down
1,2021-04-07 11:32:43.122161+00:00,56035.995,0.175507,-0.00301,Down
2,2021-04-07 11:32:44.122161+00:00,56035.995,0.175507,-0.00301,Down
3,2021-04-07 11:32:45.122161+00:00,56035.995,0.175507,-0.00301,Down
4,2021-04-07 11:32:46.122161+00:00,56035.995,0.175507,-0.00301,Down


### Chronological Split


In [3]:
feature_cols = [
    "obi_1", "obi_5", "relative_spread",
    "bid_depth_5", "ask_depth_5", "total_depth_5",
    "mid_return_1s", "mid_return_5s", "mid_return_10s",
]

train_df, val_df, test_df = time_series_train_val_test_split(df)
X_train, y_train = get_feature_target_data(train_df, feature_cols)
X_val, y_val = get_feature_target_data(val_df, feature_cols)
X_test, y_test = get_feature_target_data(test_df, feature_cols)

train_start, train_end = train_df['timestamp'].min(), train_df['timestamp'].max()
val_start, val_end = val_df['timestamp'].min(), val_df['timestamp'].max()
test_start, test_end = test_df['timestamp'].min(), test_df['timestamp'].max()
print(f"train_df: {len(train_df)}, {train_start} - {train_end}")
print(f"val_df: {len(val_df)}, {val_start} - {val_end}")
print(f"test_df: {len(test_df)}, {test_start} - {test_end}")

train_df: 721488, 2021-04-07 11:32:42.122161+00:00 - 2021-04-15 19:59:40.182190+00:00
val_df: 154605, 2021-04-15 19:59:41.182190+00:00 - 2021-04-17 14:56:36.119741+00:00
test_df: 154605, 2021-04-17 14:56:37.119741+00:00 - 2021-04-19 09:53:52.386544+00:00


### Train and Evaluate

Rule-based OBI Classifier  
Logistic Regression  
XGBoost

In [4]:
rows = []

rule_pred = rule_based_obi_predict(test_df)
rule_metrics = evaluate_predictions(y_test, rule_pred.loc[y_test.index])
rows.append({"model": "rule_based", "sample": "test", **{k: rule_metrics[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]}})

logit = train_logistic_regression(X_train, y_train)
logit_metrics = evaluate_classifier(logit, X_test, y_test)
rows.append({"model": "logistic_regression", "sample": "test", **{k: logit_metrics[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]}})

xgb = train_xgboost_classifier(X_train, y_train, X_val=X_val, y_val=y_val)
xgb_metrics = evaluate_classifier(xgb, X_test, y_test)
rows.append({"model": "xgboost", "sample": "test", **{k: xgb_metrics[k] for k in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]}})

metrics_df = pd.DataFrame(rows)
metrics_df

,model,sample,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,rule_based,test,0.426351,0.371301,0.368871,0.425315
1,logistic_regression,test,0.463439,0.375883,0.363306,0.439440
2,xgboost,test,0.474027,0.377400,0.356192,0.442166


In [ ]:
output_path = PROJECT_ROOT / "reports" / "selected_tables" / "model_metrics.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(output_path, index=False)
print(output_path)